In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

### StateBackend

In [2]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [ ]:

agent = create_deep_agent(model="openai:gpt-5.4")

agent2 = create_deep_agent(
    model="openai:gpt-5.4",
    backend=StateBackend(),
)

## Both are same -> default backend is StateBackend only

In [ ]:
# (StateBackend keeps that file inside LangGraph state)

result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'Created `/notes/todo.txt` with the exact requested content.', 'annotations': [], 'id': 'msg_0f08f80425c4a604006a213d972af081a28aa9320176a2ccbc', 'phase': 'final_answer'}]


In [ ]:
# CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-06-04T08:55:47.015116+00:00', 'modified_at': '2026-06-04T08:55:47.015116+00:00'}


In [ ]:
# Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file

followup = agent2.invoke({
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_0f08f80425c4a604006a213e1c6a9081a28478c10fdd6e5805', 'phase': 'final_answer'}]


### FileSystemBackend(local disk)


In [ ]:
# Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt, mapped onto root_dir

from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model="openai:gpt-5.4",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [ ]:
# Invoke the agent and ask it to write a file

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'Created `/notes/todo.txt` with the exact content you specified.', 'annotations': [], 'id': 'msg_036ddb9786ad9921006a213e87404081a09234d90ff35b8150', 'phase': 'final_answer'}]


In [ ]:
# Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.

fresh_agent = create_deep_agent(
    model="openai:gpt-5.4",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_080b4c2bac00a661006a213ecb2b4c81a08bf179192ace9606', 'phase': 'final_answer'}]



### StoreBackend


In [ ]:
# With StoreBackend + InMemoryStore, the file is not saved to disk at all. 
# It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="openai:gpt-5.4",
    backend=StoreBackend(
        # Local dev: static namespace
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [ ]:
import os
import uuid

# THREAD 1 — write a file
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
[{'type': 'text', 'text': 'Created `/notes/todo.txt` with the exact requested content.', 'annotations': [], 'id': 'msg_045a109ff03041fe006a213f3764ec8192b11bdea1ad59de64', 'phase': 'final_answer'}]


In [ ]:
# Thread 2: read back on a DIFFERENT thread 

thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'annotations': [], 'id': 'msg_019e6c188037b860006a213f4e68f8819eb2a97ccc9993a203', 'phase': 'final_answer'}]


: 